<a href="https://colab.research.google.com/github/safarashwan2005/w1/blob/main/w1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip uninstall -y jax jaxlib jax_cuda12_plugin jax_cuda12_pjrt dynamiqs


!pip install -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

!pip install dynamiqs

Found existing installation: jax 0.11.2
Uninstalling jax-0.11.2:
  Successfully uninstalled jax-0.11.2
Found existing installation: jaxlib 0.11.2
Uninstalling jaxlib-0.11.2:
  Successfully uninstalled jaxlib-0.11.2
Found existing installation: jax-cuda12-plugin 0.11.2
Uninstalling jax-cuda12-plugin-0.11.2:
  Successfully uninstalled jax-cuda12-plugin-0.11.2
Found existing installation: jax-cuda12-pjrt 0.11.2
Uninstalling jax-cuda12-pjrt-0.11.2:
  Successfully uninstalled jax-cuda12-pjrt-0.11.2
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached jax-0.11.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.11.2-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (1.3 kB)
  Using cached jax_cuda12_plugin-0.11.2-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (2.0 kB)
  Using cached jax_cuda12_pjrt-0.11.2-py3-none-manylinux_2_27_x86_64.whl.metadata (580 bytes)
Using cached jax_cuda12_plugin-0.11.2-cp313-cp313-manylinux_2_27_x86_64.whl (8.6 

In [2]:

!pip uninstall -y jax jaxlib jax-cuda12-pjrt jax-cuda12-plugin jax-cuda13-pjrt jax-cuda13-plugin dynamiqs


!pip install --no-cache-dir -U "jax[cuda12]" dynamiqs

Found existing installation: jax 0.11.2
Uninstalling jax-0.11.2:
  Successfully uninstalled jax-0.11.2
Found existing installation: jaxlib 0.11.2
Uninstalling jaxlib-0.11.2:
  Successfully uninstalled jaxlib-0.11.2
Found existing installation: jax-cuda12-pjrt 0.11.2
Uninstalling jax-cuda12-pjrt-0.11.2:
  Successfully uninstalled jax-cuda12-pjrt-0.11.2
Found existing installation: jax-cuda12-plugin 0.11.2
Uninstalling jax-cuda12-plugin-0.11.2:
  Successfully uninstalled jax-cuda12-plugin-0.11.2
Found existing installation: jax-cuda13-pjrt 0.11.1
Uninstalling jax-cuda13-pjrt-0.11.1:
  Successfully uninstalled jax-cuda13-pjrt-0.11.1
Found existing installation: jax-cuda13-plugin 0.11.1
Uninstalling jax-cuda13-plugin-0.11.1:
  Successfully uninstalled jax-cuda13-plugin-0.11.1
Found existing installation: dynamiqs 0.3.6
Uninstalling dynamiqs-0.3.6:
  Successfully uninstalled dynamiqs-0.3.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 kB 362.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [3]:
import jax
print(jax.devices())

[CudaDevice(id=0)]


In [4]:
!pip uninstall -y dynamiqs
!pip install git+https://github.com/dynamiqs/dynamiqs.git

Found existing installation: dynamiqs 0.3.6
Uninstalling dynamiqs-0.3.6:
  Successfully uninstalled dynamiqs-0.3.6
  Cloning https://github.com/dynamiqs/dynamiqs.git to /tmp/pip-req-build-qslghr00
  Running command git clone --filter=blob:none --quiet https://github.com/dynamiqs/dynamiqs.git /tmp/pip-req-build-qslghr00
  Resolved https://github.com/dynamiqs/dynamiqs.git to commit 3ce8982df96e15f1121e5128ce69e853fe492031
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamiqs: filename=dynamiqs-0.3.6-py3-none-any.whl size=193086 sha256=6a9121a5e883123830274ec798f66d242e02cd2d762660012feaf9ca194f154b
  Stored in directory: /tmp/pip-ephem-wheel-cache-v3la7zh9/wheels/37/b9/05/dad2d56401030647e7026c40b94de962b93e223879c99b1ab8
Successfully built dynamiqs


In [5]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import warnings
import time
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron

import jax
import jax.numpy as jnp
import dynamiqs as dq

print("Available JAX Devices:", jax.devices())
warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)

class NVCenterNearestNeighborDQ:
    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']


        self.I3 = dq.eye(3)
        self.Sz = jnp.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=jnp.complex64)
        self.Sx = (1 / jnp.sqrt(2)) * jnp.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=jnp.complex64)
        self.Sy_prime = (1 / jnp.sqrt(2)) * jnp.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=jnp.complex64)

    def _expand_operator_dense(self, op, target_index):
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = dq.tensor(result, current_op)
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = jnp.zeros((dim, dim), dtype=jnp.complex64)
        cos_z, sin_z = jnp.cos(self.zeta), jnp.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_dense(self.Sx, j)
            Sy_p_j = self._expand_operator_dense(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)
        return H_control

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = jnp.zeros((dim, dim), dtype=jnp.complex64)
        Sz_sq = self.Sz @ self.Sz

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_dense(Sz_sq, j)
            Sz_j = self._expand_operator_dense(self.Sz, j)
            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_dense(self.Sz, j)
            Sz_j_next = self._expand_operator_dense(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * (Sz_j @ Sz_j_next)

        return H_drift

    def get_ground_state(self):
        ms_0 = jnp.array([[0], [1], [0]], dtype=jnp.complex64)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = dq.tensor(ground_state, ms_0)
        return ground_state

    def evolve_state(
        self, initial_state, t_total, control_pulse_func, time_steps=2000
    ):


        H_drift = self.get_drift_hamiltonian()
        H_control = self.get_control_hamiltonian()

        H_t = H_drift + dq.modulated(control_pulse_func, H_control)

        tsave = jnp.linspace(0, t_total, time_steps)
        dt = t_total / time_steps

        result = dq.sesolve(H_t, initial_state, tsave, method=dq.method.Euler(dt=dt))

        # Robust unboxing across different dynamiqs versions
        final_state = result.states[-1]
        if hasattr(final_state, 'to_jax'):
            return final_state.to_jax()
        else:
            return dq.to_jax(final_state) if hasattr(dq, 'to_jax') else jnp.asarray(final_state)


if __name__ == "__main__":
    start_time = time.perf_counter()
    N_test = 8
    test_params = {
        'N': N_test,
        'D': [2870.0] * N_test,
        'omega_0': 2870.0,
        'gamma_e': 28000.0,
        'Bz': [1e-3] * N_test,
        'zeta': np.pi / 4,
        'g': [0.1] * (N_test - 1),
    }

    print(f"Building Hamiltonians for N={N_test} on CPU...")
    nv_chain = NVCenterNearestNeighborDQ(test_params)
    initial_state_cpu = nv_chain.get_ground_state()
    t_pulse = 1.0

    print("\n--- Testing Lambda Function Pulse ---")
    def lambda_pulse(t):
        return 2.0 * jnp.sin(jnp.pi * (t / t_pulse))

    final_state_lambda = nv_chain.evolve_state(
        initial_state=initial_state_cpu,
        t_total=t_pulse,
        control_pulse_func=lambda_pulse,
        time_steps=2000,
    )

    norm_lambda = jnp.linalg.norm(final_state_lambda)
    print("Norm check (Lambda):", norm_lambda)

    print("\n--- Testing Array Pulse ---")
    time_points = jnp.linspace(0, t_pulse, 100)
    array_pulse = 2.0 * jnp.sin(jnp.pi * (time_points / t_pulse))

    def array_interp_pulse(t):
        return jnp.interp(t, time_points, array_pulse)

    final_state_array = nv_chain.evolve_state(
        initial_state=initial_state_cpu,
        t_total=t_pulse,
        control_pulse_func=array_interp_pulse,
        time_steps=2000,
    )

    norm_array = jnp.linalg.norm(final_state_array)
    print("Norm check (Array):", norm_array)

    end_time = time.perf_counter()
    print(f"Time taken: {end_time - start_time:.4f} seconds")

Available JAX Devices: [CudaDevice(id=0)]
Building Hamiltonians for N=8 on CPU...

--- Testing Lambda Function Pulse ---


|██████████| 100.0% ◆ elapsed 5.82s ◆ remaining 0.00ms


Norm check (Lambda): 1.000046

--- Testing Array Pulse ---


|██████████| 100.0% ◆ elapsed 5.99s ◆ remaining 0.00ms

Norm check (Array): 1.000046
Time taken: 44.4607 seconds
